## CS 4412 Data Mining 
#### M2 - Initial Implementation
#### Victor Urey

There are two discovery questions being analyzed:

1. How do hardware features correlate with price tiers across popular smartphone brands?

2. What hardware patterns exist among the most popular phone models sold over time?

Investigation of these questions will be using two data mining techniques:

- Association Rule Mining
- Clustering Analysis

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn mlxtend

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from mlxtend.frequent_patterns import apriori, association_rules

## 1. Dataset 1: Smartphone Specifications Dataset

This dataset contains nearly 1000 smartphone models and their hardware specifications including display size, camera resolution, included RAM, and price at release.

In [ ]:
phoneSpecs = pd.read_csv("smartphone specifications dataset.csv")
phoneSpecs.head()

## Dataset 2: World's Best-Selling Phones

This dataset contains sales information for the top-selling phone models over the past three decades. It includes information on the brand, model, release year, and units sold to date.

In [ ]:
phoneSales = pd.read_csv("top 120 best-selling mobile phones.csv")
phoneSales.head()

In [ ]:
phoneSpecs.info()

phoneSales.info()

In [ ]:
phoneSpecs.describe()

In [ ]:
phoneSales.describe()

Before analyzing these datasets there are a few values that must be cleaned:
- Missing refresh rate values
- Incosistent phone model naming
- Possible duplicates (or variations of the same model)

In [ ]:
phoneSpecs.isnull().sum()

In [ ]:
phoneSpecs = phoneSpecs.dropna(subset=['model', 'price'])

In [ ]:
phoneSales = phoneSales.drop_duplicates()

Since these two datasets refer to phone models and manufacturer info, it's important to standardize these names before merging:
- Convert text names to lowercase
- Remove extra spaces
- Standardize common brand naming
- Manage punctuation (parathesis, periods, etc.)

Moreover, the Phone Specifications Dataset includes both the brand and the model information in the same cell (e.g. iPhone 14 Pro-Max). Solving this requires that we:
1. Split the brand and model names (iPhone, 14 Pro-Max)
2. Clean special characters (Pro-Max -> Pro Max)
3. Merge the new expanded dataset with the Best-Selling Dataset 


In [ ]:
# Split brand and model names into two columns
# Standardize names into lowercase
def splitBrandModel(text):
    text = str(text).lower().strip()
    piece = text.split(" ", 1)
    return piece[0], piece[1]

# Create new columns (brand & model)
phoneSpecs["brand"], phoneSpecs["model"] = zip(*phoneSpecs["model"].apply(splitBrandModel)) 

# Moves new brand column to column 0
column = phoneSpecs.pop("brand")
phoneSpecs.insert(0, "brand", column)

# Remove duplicate prices per model, keep median price
median = phoneSpecs.groupby(['brand', 'model'])['price'].median().reset_index()
median = median.rename(columns={'price': 'median price'})
phoneSpecs = phoneSpecs.merge(median, on=['brand', 'model'])
phoneSpecs = phoneSpecs[phoneSpecs['price'] == phoneSpecs['median price']]

phoneSpecs = phoneSpecs.drop(columns='median price')
phoneSpecs = phoneSpecs.reset_index(drop=True)

phoneSpecs.head()

In [ ]:
# Clean Best-Selling Phones dataset
phoneSales = phoneSales.rename(columns={"Rank": "rank", "Manufacturer": "brand", "Model": "model", "Form Factor": "form factor", "Smartphone?": "smartphone?", "Year": "year", "Units Sold (million )": "sold units (millions)"})

phoneSales["brand"] = phoneSales["brand"].str.lower().str.strip()
phoneSales["model"] = phoneSales["model"].str.lower().str.strip()

# Remove special characters
phoneSales["model"] = phoneSales["model"].str.replace('-', ' ', regex=False)

phoneSales.head()

In [ ]:
# Split "&" in Best-Selling Phones dataset
phoneSales = phoneSales.assign(model=phoneSales['model'].str.split('&')).explode('model')
phoneSales['model'] = phoneSales['model'].str.strip()

phoneSales = phoneSales.assign(model=phoneSales['model'].str.split(',')).explode('model')
phoneSales['model'] = phoneSales['model'].str.strip()

# Add spaces between numbers and model (iphone11 -> iphone 11)
def numberSpacing(row):
    if "e11" in row['model'].lower():
        return 'iphone 11'
    elif "11pro max" in row['model'].lower():
        return 'iphone 11 pro max'
    elif "11pro" in row['model'].lower():
        return 'iphone 11 pro'
    return row['model']
phoneSales['model'] = phoneSales.apply(numberSpacing, axis=1)

# Add "iphone" to new rows
def addiPhone(row):
    if row['brand'].lower() == 'apple' and not row['model'].lower().startswith('iphone'):
        return 'iphone ' + row['model']
    return row['model']
phoneSales['model'] = phoneSales.apply(addiPhone, axis=1)
    
phoneSales.head()

In [ ]:
mergedDataset = pd.merge(
    phoneSales,
    phoneSpecs,
    on=['brand', 'model'],
    how="left" # Keeps all etnries from Best-Selling Phones Dataset
)

# Moves new rank column to column 0
column = mergedDataset.pop("rank")
mergedDataset.insert(0, "rank", column)

print("Merged dataset shape:", mergedDataset.shape)
print(mergedDataset)